In [ ]:
import numpy as np
import cupy as cp

from cuml.svm import SVC
from cuml.preprocessing import StandardScaler
from cuml.metrics import accuracy_score

# =====================================================
# 1. Load FEATURES + LABELS
# =====================================================

# Train set (50,000)
X_train = np.fromfile(
    "/content/output/train_features.bin",
    dtype=np.float32
).reshape(50000, 8192)

y_train = np.fromfile(
    "/content/output/train_labels.bin",
    dtype=np.uint8
)

# Test set (10,000)
X_test = np.fromfile(
    "/content/output/test_features.bin",
    dtype=np.float32
).reshape(10000, 8192)

y_test = np.fromfile(
    "/content/output/test_labels.bin",
    dtype=np.uint8
)

print("Train X:", X_train.shape, "Train y:", y_train.shape)
print("Test  X:", X_test.shape,  "Test  y:", y_test.shape)

# =====================================================
# 2. CHUYỂN SANG GPU
# =====================================================

X_train = cp.asarray(X_train)
X_test  = cp.asarray(X_test)
y_train = cp.asarray(y_train)
y_test  = cp.asarray(y_test)

# =====================================================
# 3. STANDARD SCALER
# =====================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# =====================================================
# 4. TRAIN SVM
# =====================================================

svm = SVC(
    kernel="rbf",
    C=10,
    gamma="auto"
)

svm.fit(X_train, y_train)

# =====================================================
# 5. EVALUATION
# =====================================================

y_pred = svm.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("SVM Test Accuracy:", float(acc))

In [ ]:
import numpy as np

np.savez(
    "svm_cuml_model.npz",
    support_vectors=svm.support_vectors_,
    dual_coef=svm.dual_coef_,
    intercept=svm.intercept_,
    classes=svm.classes_
)

print("SVM model saved to svm_cuml_model.npz")